In [0]:

# import json
# import re
# import time
# from collections import Counter, defaultdict
# from datetime import datetime, timedelta, timezone

# import requests
# from requests.auth import HTTPBasicAuth

# # --------------------------------------------------
# # CONFIGURATION
# # --------------------------------------------------

# SECRET_SCOPE = "jira"                                               # secret scope name
# JIRA_BASE_URL = dbutils.secrets.get(SECRET_SCOPE, "base_url").rstrip("/")  # https://<site>.atlassian.net
# JIRA_EMAIL = dbutils.secrets.get(SECRET_SCOPE, "email")
# JIRA_API_TOKEN = dbutils.secrets.get(SECRET_SCOPE, "api_token")

# PROJECT_KEYS = []          # e.g. ["KAFKA"]; empty list = all projects visible to the user

# # S3 target. Either the external Volume on S3 used by cell 1, or an s3:// URI
# # covered by a Unity Catalog external location you can WRITE FILES to.
# TARGET_DIR = "/Volumes/team_8/quantum_force/quantum_force"
# BACKUP_EXISTING = True     # copy current JSON files to _backup/<run_ts>/ before overwrite

# # Jira custom fields: None = auto-detect by field name (GET /rest/api/3/field)
# FIELD_SPRINT = None        # usually customfield_10020 ("Sprint")
# FIELD_STORY_POINTS = None  # "Story point estimate" / "Story Points"
# FIELD_TEAM = None          # "Team" (Atlassian Teams field)

# EPIC_ISSUE_TYPE = "Epic"
# FEATURE_ISSUE_TYPE = "Feature"
# IGNORE_ROLES = {"atlassian-addons-project-access"}
# DEFAULT_CAPACITY_POINTS = 10       # Jira has no capacity field; used for developers.json
# GENERATE_DIM_FILES = False         # True = regenerate dim_calendar / dim_time (not Jira data)
# DATA_TYPE = "jira_api"
# PAGE_SIZE = 100
# RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

# # --------------------------------------------------
# # HTTP HELPERS (retry on 429 / 5xx, pagination)
# # --------------------------------------------------

# session = requests.Session()
# session.auth = HTTPBasicAuth(JIRA_EMAIL, JIRA_API_TOKEN)
# session.headers.update({"Accept": "application/json", "Content-Type": "application/json"})


# def jira(method, path, params=None, body=None, retries=6):
#     url = f"{JIRA_BASE_URL}{path}"
#     for attempt in range(retries):
#         r = session.request(method, url, params=params, json=body, timeout=60)
#         if r.status_code == 429 or r.status_code >= 500:
#             time.sleep(int(r.headers.get("Retry-After", 2 ** attempt)))
#             continue
#         if r.status_code >= 400:
#             raise RuntimeError(f"Jira {method} {path} -> {r.status_code}: {r.text[:500]}")
#         return r.json() if r.text else {}
#     raise RuntimeError(f"Jira {method} {path} failed after {retries} retries")


# def paged(path, params=None, values_key="values"):
#     # startAt/maxResults pagination (platform + agile APIs)
#     params, start, out = dict(params or {}), 0, []
#     while True:
#         params.update({"startAt": start, "maxResults": PAGE_SIZE})
#         data = jira("GET", path, params=params)
#         vals = data.get(values_key, []) if isinstance(data, dict) else data
#         out.extend(vals)
#         if not vals:
#             break
#         if isinstance(data, dict):
#             if data.get("isLast") is True:
#                 break
#             if "total" in data and start + len(vals) >= data["total"]:
#                 break
#             if "isLast" not in data and "total" not in data and len(vals) < PAGE_SIZE:
#                 break
#         elif len(vals) < PAGE_SIZE:
#             break
#         start += len(vals)
#     return out


# def search_issues(jql, fields):
#     # Enhanced JQL search (nextPageToken pagination)
#     out, token = [], None
#     while True:
#         body = {"jql": jql, "fields": fields, "maxResults": PAGE_SIZE}
#         if token:
#             body["nextPageToken"] = token
#         data = jira("POST", "/rest/api/3/search/jql", body=body)
#         out.extend(data.get("issues", []))
#         token = data.get("nextPageToken")
#         if not token or data.get("isLast"):
#             break
#     return out


# def fetch_status_changelogs(issue_ids):
#     # Bulk changelog API: status changes only, 1000 issues per request
#     histories = defaultdict(list)
#     for i in range(0, len(issue_ids), 1000):
#         chunk, token = issue_ids[i:i + 1000], None
#         while True:
#             body = {"issueIdsOrKeys": chunk, "fieldIds": ["status"], "maxResults": 1000}
#             if token:
#                 body["nextPageToken"] = token
#             data = jira("POST", "/rest/api/3/changelog/bulkfetch", body=body)
#             for log in data.get("issueChangeLogs", []):
#                 histories[str(log["issueId"])].extend(log.get("changeHistories", []))
#             token = data.get("nextPageToken")
#             if not token:
#                 break
#     return histories

# # --------------------------------------------------
# # VALUE HELPERS
# # --------------------------------------------------

# def parse_ts(v):
#     if v in (None, ""):
#         return None
#     if isinstance(v, (int, float)):
#         return datetime.fromtimestamp(v / 1000, tz=timezone.utc)
#     s = str(v).replace("Z", "+00:00")
#     s = re.sub(r"([+-]\d{2})(\d{2})$", r"\1:\2", s)
#     d = datetime.fromisoformat(s)
#     return d if d.tzinfo else d.replace(tzinfo=timezone.utc)


# def ts_str(v):
#     d = parse_ts(v)
#     return d.astimezone(timezone.utc).strftime("%Y-%m-%d %H:%M:%S") if d else None


# def date_str(v):
#     d = parse_ts(v)
#     return d.astimezone(timezone.utc).strftime("%Y-%m-%d") if d else None


# def date_key(v):
#     d = parse_ts(v)
#     return int(d.astimezone(timezone.utc).strftime("%Y%m%d")) if d else None


# def pick_sprint(value):
#     # Issue sprint field is a list; prefer the active sprint, else the latest one
#     if not value:
#         return None
#     value = value if isinstance(value, list) else [value]
#     sprints = [s for s in value if isinstance(s, dict) and s.get("id") is not None]
#     if not sprints:
#         return None
#     active = [s for s in sprints if s.get("state") == "active"]
#     return (active or sprints)[-1]


# def pick_team(value):
#     if not value:
#         return None, None
#     if isinstance(value, list):
#         return pick_team(value[0])
#     if isinstance(value, dict):
#         tid = value.get("id") or value.get("value")
#         name = value.get("name") or value.get("title") or value.get("value")
#         return (str(tid) if tid else None), name
#     return str(value), str(value)


# def to_float(v):
#     try:
#         return float(v) if v is not None else None
#     except (TypeError, ValueError):
#         return None


# def dedup(rows, keys):
#     seen, out = set(), []
#     for r in rows:
#         k = tuple(r.get(c) for c in keys)
#         if k not in seen:
#             seen.add(k)
#             out.append(r)
#     return out

# # --------------------------------------------------
# # 1. FIELD DISCOVERY
# # --------------------------------------------------

# fields_meta = jira("GET", "/rest/api/3/field")


# def find_field(current, names):
#     if current:
#         return current
#     for f in fields_meta:
#         if f.get("name", "").strip().lower() in names:
#             return f["id"]
#     return None


# FIELD_SPRINT = find_field(FIELD_SPRINT, {"sprint"})
# FIELD_STORY_POINTS = find_field(FIELD_STORY_POINTS, {"story point estimate", "story points"})
# FIELD_TEAM = find_field(FIELD_TEAM, {"team"})
# print(f"Sprint field: {FIELD_SPRINT} | Story points: {FIELD_STORY_POINTS} | Team: {FIELD_TEAM}")

# # --------------------------------------------------
# # 2. PROJECTS
# # --------------------------------------------------

# project_params = {"expand": "description"}
# if PROJECT_KEYS:
#     project_params["keys"] = PROJECT_KEYS
# projects_raw = paged("/rest/api/3/project/search", project_params)
# if not projects_raw:
#     raise ValueError("No Jira projects returned. Check PROJECT_KEYS and the user's permissions.")

# projects = [{
#     "project_id": str(p["id"]),
#     "project_key": p["key"],
#     "project_name": p.get("name"),
#     "description": p.get("description"),
#     "project_type": p.get("projectTypeKey"),
#     "source_project_key": p["key"],
#     "data_type": DATA_TYPE,
# } for p in projects_raw]
# project_by_id = {p["project_id"]: p for p in projects}
# print(f"Projects: {len(projects)}")

# # --------------------------------------------------
# # 3. ISSUES (incl. epics and features)
# # --------------------------------------------------

# issue_fields = [f for f in [
#     "summary", "issuetype", "status", "priority", "resolution", "resolutiondate",
#     "created", "updated", "assignee", "reporter", "project", "parent",
#     FIELD_SPRINT, FIELD_STORY_POINTS, FIELD_TEAM,
# ] if f]

# keys_jql = ",".join(f'"{p["project_key"]}"' for p in projects)
# raw_issues = search_issues(f"project in ({keys_jql}) ORDER BY created ASC", issue_fields)
# if not raw_issues:
#     raise ValueError("No issues returned from Jira.")
# print(f"Issues fetched (all types): {len(raw_issues)}")

# by_id = {str(i["id"]): i for i in raw_issues}


# def f_(issue, name):
#     return issue["fields"].get(name) if name else None


# def itype(issue):
#     return ((issue["fields"].get("issuetype") or {}).get("name") or "")


# def parent_of(issue):
#     pid = (issue["fields"].get("parent") or {}).get("id")
#     return by_id.get(str(pid)) if pid else None


# def resolve_hierarchy(issue, depth=0):
#     # Returns (epic_id, feature_id) by walking up the parent chain
#     parent = parent_of(issue)
#     if not parent or depth > 3:
#         return None, None
#     ptype = itype(parent)
#     if ptype == EPIC_ISSUE_TYPE:
#         return str(parent["id"]), None
#     if ptype == FEATURE_ISSUE_TYPE:
#         grand = parent_of(parent)
#         epic = str(grand["id"]) if grand and itype(grand) == EPIC_ISSUE_TYPE else None
#         return epic, str(parent["id"])
#     return resolve_hierarchy(parent, depth + 1)


# epics, features, work_issues = [], [], []
# for i in raw_issues:
#     t = itype(i)
#     proj = i["fields"]["project"]
#     if t == EPIC_ISSUE_TYPE:
#         epics.append({
#             "epic_id": str(i["id"]), "epic_key": i["key"],
#             "epic_name": f_(i, "summary"),
#             "project_id": str(proj["id"]), "project_key": proj["key"],
#             "status": (f_(i, "status") or {}).get("name"),
#             "created_date": date_str(f_(i, "created")),
#             "data_type": DATA_TYPE,
#         })
#     elif t == FEATURE_ISSUE_TYPE:
#         epic_id, _ = resolve_hierarchy(i)
#         features.append({
#             "feature_id": str(i["id"]), "feature_key": i["key"],
#             "feature_name": f_(i, "summary"), "epic_id": epic_id,
#             "project_id": str(proj["id"]),
#             "status": (f_(i, "status") or {}).get("name"),
#             "data_type": DATA_TYPE,
#         })
#     else:
#         work_issues.append(i)

# print(f"Epics: {len(epics)} | Features: {len(features)} | Work issues: {len(work_issues)}")

# # --------------------------------------------------
# # 4. STATUS HISTORY (changelog)
# # --------------------------------------------------

# changelogs = fetch_status_changelogs([str(i["id"]) for i in work_issues])

# issue_history, first_change = [], {}
# for issue_id, hist in changelogs.items():
#     seq = 0
#     for h in sorted(hist, key=lambda x: parse_ts(x.get("created")) or datetime.min.replace(tzinfo=timezone.utc)):
#         for item in h.get("items", []):
#             if item.get("field") != "status":
#                 continue
#             seq += 1
#             changed_at = ts_str(h.get("created"))
#             first_change.setdefault(issue_id, h.get("created"))
#             issue_history.append({
#                 "history_id": f"{issue_id}-{h.get('id')}-{seq}",
#                 "issue_id": issue_id,
#                 "from_status": item.get("fromString"),
#                 "status": item.get("toString"),
#                 "status_changed_at": changed_at,
#                 "sequence_number": seq,
#                 "changed_by": (h.get("author") or {}).get("displayName"),
#                 "data_type": DATA_TYPE,
#             })
# print(f"Status transitions: {len(issue_history)}")

# # --------------------------------------------------
# # 5. SPRINTS (scrum boards + any sprint referenced by an issue)
# # --------------------------------------------------

# sprint_raw = {}   # sprint_id -> (sprint json, project_id)
# for p in projects:
#     boards = paged("/rest/agile/1.0/board", {"projectKeyOrId": p["project_key"]})
#     for b in boards:
#         if b.get("type") != "scrum":
#             continue
#         for s in paged(f"/rest/agile/1.0/board/{b['id']}/sprint"):
#             sprint_raw.setdefault(str(s["id"]), (s, p["project_id"]))

# issue_sprint = {}
# for i in work_issues:
#     s = pick_sprint(f_(i, FIELD_SPRINT))
#     if s:
#         issue_sprint[str(i["id"])] = s
#         if str(s["id"]) not in sprint_raw:
#             full = jira("GET", f"/rest/agile/1.0/sprint/{s['id']}")
#             sprint_raw[str(s["id"])] = (full, str(i["fields"]["project"]["id"]))

# sprints = []
# by_project = defaultdict(list)
# for sid, (s, pid) in sprint_raw.items():
#     by_project[pid].append((sid, s))
# for pid, items in by_project.items():
#     items.sort(key=lambda x: (parse_ts(x[1].get("startDate")) or datetime.max.replace(tzinfo=timezone.utc), int(x[0])))
#     for n, (sid, s) in enumerate(items, start=1):
#         start, end = parse_ts(s.get("startDate")), parse_ts(s.get("endDate"))
#         sprints.append({
#             "sprint_id": sid,
#             "sprint_name": s.get("name"),
#             "sprint_number": n,
#             "start_date": date_str(s.get("startDate")),
#             "end_date": date_str(s.get("endDate")),
#             "duration_days": (end - start).days if start and end else None,
#             "sprint_type": "scrum",
#             "status": s.get("state"),
#             "goal": s.get("goal"),
#             "project_id": pid,
#             "start_date_key": date_key(s.get("startDate")),
#             "end_date_key": date_key(s.get("endDate")),
#             "data_type": DATA_TYPE,
#         })
# print(f"Sprints: {len(sprints)}")

# # --------------------------------------------------
# # 6. ISSUES_ENRICHED
# # --------------------------------------------------

# issues_enriched, teams_seen = [], {}
# person_names, person_teams = {}, defaultdict(list)
# assignees = set()

# for i in work_issues:
#     fld, iid = i["fields"], str(i["id"])
#     proj = fld["project"]
#     assignee = fld.get("assignee") or {}
#     reporter = fld.get("reporter") or {}
#     team_id, team_name = pick_team(f_(i, FIELD_TEAM))
#     epic_id, feature_id = resolve_hierarchy(i)
#     sprint = issue_sprint.get(iid)

#     created, resolved = parse_ts(fld.get("created")), parse_ts(fld.get("resolutiondate"))
#     work_start = parse_ts(first_change.get(iid)) or created
#     cycle = round(max((resolved - work_start).total_seconds(), 0) / 86400, 2) if resolved and work_start else None

#     acc = assignee.get("accountId")
#     if acc:
#         assignees.add(acc)
#         person_names[acc] = assignee.get("displayName")
#         if team_id:
#             person_teams[acc].append(team_id)
#     if reporter.get("accountId"):
#         person_names.setdefault(reporter["accountId"], reporter.get("displayName"))
#     if team_id:
#         teams_seen[team_id] = team_name

#     issues_enriched.append({
#         "issue_id": iid,
#         "issue_key": i["key"],
#         "project_id": str(proj["id"]),
#         "project_key": proj["key"],
#         "issue_type": itype(i),
#         "summary": fld.get("summary"),
#         "status": (fld.get("status") or {}).get("name"),
#         "status_category": ((fld.get("status") or {}).get("statusCategory") or {}).get("name"),
#         "priority": (fld.get("priority") or {}).get("name"),
#         "resolution": (fld.get("resolution") or {}).get("name"),
#         "created": ts_str(fld.get("created")),
#         "updated": ts_str(fld.get("updated")),
#         "resolution_date": ts_str(fld.get("resolutiondate")),
#         "is_resolved": resolved is not None,
#         "story_points": to_float(f_(i, FIELD_STORY_POINTS)),
#         "cycle_time_days": cycle,
#         "assignee_person_id": acc,
#         "developer_id": acc,
#         "reporter": reporter.get("displayName"),
#         "team_id": team_id,
#         "team": team_name,
#         "sprint_id": str(sprint["id"]) if sprint else None,
#         "sprint_name": sprint.get("name") if sprint else None,
#         "epic_id": epic_id,
#         "feature_id": feature_id,
#         "data_type": DATA_TYPE,
#     })

# # --------------------------------------------------
# # 7. ROLES AND PROJECT ROLE MEMBERSHIP
# # --------------------------------------------------

# roles = [{
#     "role_id": str(r["id"]), "role_name": r.get("name"),
#     "description": r.get("description"), "data_type": DATA_TYPE,
# } for r in jira("GET", "/rest/api/3/role") if r.get("name") not in IGNORE_ROLES]
# role_ids = {r["role_id"] for r in roles}

# person_role = {}
# for p in projects:
#     try:
#         role_links = jira("GET", f"/rest/api/3/project/{p['project_key']}/role")
#     except RuntimeError as e:
#         print(f"WARNING: cannot read roles for {p['project_key']}: {str(e)[:200]}")
#         continue
#     for role_name, url in role_links.items():
#         if role_name in IGNORE_ROLES:
#             continue
#         role_id = url.rstrip("/").split("/")[-1]
#         detail = jira("GET", f"/rest/api/3/project/{p['project_key']}/role/{role_id}")
#         for actor in detail.get("actors", []):
#             acc = (actor.get("actorUser") or {}).get("accountId")
#             if acc:
#                 person_role.setdefault(acc, (str(detail["id"]), detail.get("name")))
#                 person_names.setdefault(acc, actor.get("displayName"))

# # --------------------------------------------------
# # 8. TEAMS, PEOPLE, DEVELOPERS AND BRIDGES
# # --------------------------------------------------

# teams = [{
#     "team_id": tid, "team_name": name, "team_type": "jira_team",
#     "description": None, "data_type": DATA_TYPE,
# } for tid, name in teams_seen.items()]

# people, developers, team_members, user_team_access = [], [], [], []
# for acc, name in person_names.items():
#     team_id = Counter(person_teams[acc]).most_common(1)[0][0] if person_teams.get(acc) else None
#     role_id, role_name = person_role.get(acc, (None, None))
#     if role_id not in role_ids:
#         role_id, role_name = None, None
#     is_dev = acc in assignees
#     people.append({
#         "person_id": acc, "person_name": name,
#         "team_id": team_id, "team": teams_seen.get(team_id),
#         "role_id": role_id, "role_name": role_name,
#         "person_type": "developer" if is_dev else "user",
#         "developer_id": acc if is_dev else None,
#         "data_type": DATA_TYPE,
#     })
#     if is_dev:
#         developers.append({
#             "developer_id": acc, "person_id": acc, "developer_name": name,
#             "team_id": team_id, "capacity_points_per_sprint": DEFAULT_CAPACITY_POINTS,
#             "data_type": DATA_TYPE,
#         })
#     if team_id:
#         team_members.append({
#             "team_member_id": f"{team_id}-{acc}", "team_id": team_id,
#             "person_id": acc, "data_type": DATA_TYPE,
#         })
#         user_team_access.append({
#             "access_id": f"{acc}-{team_id}", "person_id": acc, "team_id": team_id,
#             "role_id": role_id, "access_level": "member", "data_type": DATA_TYPE,
#         })

# team_roles = dedup([{
#     "team_role_id": f"{p['team_id']}-{p['role_id']}", "team_id": p["team_id"],
#     "role_id": p["role_id"], "data_type": DATA_TYPE,
# } for p in people if p["team_id"] and p["role_id"]], ["team_id", "role_id"])

# project_sprint_bridge = dedup(
#     [{"project_id": s["project_id"], "sprint_id": s["sprint_id"], "data_type": DATA_TYPE} for s in sprints] +
#     [{"project_id": r["project_id"], "sprint_id": r["sprint_id"], "data_type": DATA_TYPE}
#      for r in issues_enriched if r["sprint_id"]],
#     ["project_id", "sprint_id"],
# )

# # --------------------------------------------------
# # 9. OPTIONAL CALENDAR / TIME DIMENSIONS (not Jira data)
# # --------------------------------------------------

# outputs = {
#     "projects": (projects, ["project_id"]),
#     "teams": (teams, ["team_id"]),
#     "roles": (roles, ["role_id"]),
#     "team_roles": (team_roles, ["team_id", "role_id"]),
#     "developers": (developers, ["developer_id"]),
#     "people": (people, ["person_id"]),
#     "team_members": (team_members, ["team_id", "person_id"]),
#     "user_team_access": (user_team_access, ["access_id"]),
#     "epics": (epics, ["epic_id"]),
#     "features": (features, ["feature_id"]),
#     "sprints": (sprints, ["sprint_id"]),
#     "project_sprint_bridge": (project_sprint_bridge, ["project_id", "sprint_id"]),
#     "issues_enriched": (issues_enriched, ["issue_id"]),
#     "issue_history": (issue_history, ["history_id"]),
# }

# if GENERATE_DIM_FILES:
#     created_dates = [parse_ts(r["created"] + "Z") for r in issues_enriched if r["created"]]
#     d, end = min(created_dates).date(), (max(created_dates) + timedelta(days=365)).date()
#     dim_calendar = []
#     while d <= end:
#         dim_calendar.append({
#             "date_key": int(d.strftime("%Y%m%d")), "date": d.isoformat(),
#             "year": d.year, "quarter": (d.month - 1) // 3 + 1, "month": d.month,
#             "month_name": d.strftime("%B"), "day": d.day,
#             "day_of_week": d.isoweekday(), "day_name": d.strftime("%A"),
#             "week_of_year": d.isocalendar()[1], "is_weekend": d.isoweekday() >= 6,
#         })
#         d += timedelta(days=1)
#     dim_time = [{
#         "time_key": h * 100 + m, "hour": h, "minute": m, "time_label": f"{h:02d}:{m:02d}",
#     } for h in range(24) for m in range(60)]
#     outputs["dim_calendar"] = (dim_calendar, ["date_key"])
#     outputs["dim_time"] = (dim_time, ["time_key"])
# else:
#     print("dim_calendar / dim_time not regenerated: existing files in TARGET_DIR are kept.")

# # --------------------------------------------------
# # 10. VALIDATE EVERYTHING BEFORE WRITING (no partial overwrite)
# # --------------------------------------------------

# problems = []
# for name, (rows, keys) in outputs.items():
#     key_vals = [tuple(r.get(k) for k in keys) for r in rows]
#     if any(v is None for kv in key_vals for v in kv):
#         problems.append(f"{name}: NULL key {keys}")
#     if len(key_vals) != len(set(key_vals)):
#         problems.append(f"{name}: duplicate key {keys}")

# sprint_ids = {s["sprint_id"] for s in sprints}
# person_ids = {p["person_id"] for p in people}
# issue_ids = {r["issue_id"] for r in issues_enriched}
# orphans = {
#     "issues -> projects": sum(r["project_id"] not in project_by_id for r in issues_enriched),
#     "issues -> sprints": sum(bool(r["sprint_id"]) and r["sprint_id"] not in sprint_ids for r in issues_enriched),
#     "issues -> people": sum(bool(r["assignee_person_id"]) and r["assignee_person_id"] not in person_ids for r in issues_enriched),
#     "history -> issues": sum(h["issue_id"] not in issue_ids for h in issue_history),
# }
# problems += [f"orphans {k}: {v}" for k, v in orphans.items() if v]
# if problems:
#     raise ValueError("Extract validation failed, nothing written: " + "; ".join(problems))

# no_team = sum(1 for r in issues_enriched if r["sprint_id"] and not r["team_id"])
# if no_team:
#     print(f"WARNING: {no_team} sprint issues have no team. Gold fact_sprint_delivery requires team_id.")

# # --------------------------------------------------
# # 11. WRITE JSON FILES TO S3 (one JSON array per file)
# # --------------------------------------------------

# target_dir = TARGET_DIR.rstrip("/")

# if BACKUP_EXISTING:
#     try:
#         existing = [f for f in dbutils.fs.ls(target_dir) if f.name.lower().endswith(".json")]
#     except Exception:
#         existing = []
#     for f in existing:
#         if f.name[:-5].lower() in outputs:
#             dbutils.fs.cp(f.path, f"{target_dir}/_backup/{RUN_TS}/{f.name}")
#     if existing:
#         print(f"Backed up existing files to {target_dir}/_backup/{RUN_TS}/")


# def write_json(name, rows):
#     path = f"{target_dir}/{name}.json"
#     payload = json.dumps(rows, default=str, ensure_ascii=False)
#     if path.startswith("/Volumes/"):
#         with open(path, "w", encoding="utf-8") as fh:
#             fh.write(payload)
#     else:
#         dbutils.fs.put(path, payload, overwrite=True)
#     return path


# summary = []
# for name, (rows, _) in outputs.items():
#     if not rows:
#         print(f"WARNING: {name} is empty in Jira; existing {name}.json left unchanged.")
#         summary.append((name, 0, "SKIPPED (empty)"))
#         continue
#     summary.append((name, len(rows), write_json(name, rows)))

# display(spark.createDataFrame(summary, "file STRING, rows LONG, path STRING"))
# print("\nJIRA EXTRACT COMPLETED. Run cell 1 to load these files into Bronze.")